## final system eval

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()

FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)


benchmark = pd.read_csv(
    FINAL_FOLDER
    / "final_60_query_benchmark.csv"
)

single = pd.read_csv(
    FINAL_FOLDER
    / "final_single_agent_results.csv"
)

no_coord = pd.read_csv(
    FINAL_FOLDER
    / "final_no_coordination_results.csv"
)

coord = pd.read_csv(
    FINAL_FOLDER
    / "final_coordinated_results.csv"
)


print(
    len(benchmark),
    len(single),
    len(no_coord),
    len(coord)
)

60 60 60 60


In [2]:
print(
    "Single errors:",
    single["error"].notna().sum()
)

print(
    "No coordination errors:",
    no_coord["error"].notna().sum()
)

print(
    "Coordinated errors:",
    coord["error"].notna().sum()
)

Single errors: 0
No coordination errors: 4
Coordinated errors: 4


In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


vector_db = FAISS.load_local(
    str(
        PROJECT_ROOT
        / "Data"
        / "Cleaned"
        / "faiss_index"
    ),
    embedding_model,
    allow_dangerous_deserialization=True
)


documents = list(
    vector_db.docstore._dict.values()
)


asin_lookup = {}


for document in documents:

    asin = str(
        document.metadata.get(
            "parent_asin",
            ""
        )
    )


    product = dict(
        document.metadata
    )

    product[
        "product_text"
    ] = document.page_content

    product[
        "description"
    ] = document.page_content


    if asin:

        asin_lookup[
            asin
        ] = product


print(
    "Amazon products available:",
    len(asin_lookup)
)

C:\Users\srush\AppData\Local\Temp\ipykernel_15132\3377126206.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Amazon products available: 10000


In [4]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(
        str(PROJECT_ROOT)
    )


from Agents.query_agent import (
    QueryAgent
)

from Agents.verifier_agent import (
    check_brand_match,
    check_product_type_match,
    check_feature_match,
    normalise_price
)


query_agent = QueryAgent()

In [5]:
def split_asins(value):

    if pd.isna(value):
        return []

    return [
        item.strip()
        for item in str(value).split("|")
        if item.strip()
    ]

In [6]:
def evaluate_product(
    structured_query,
    asin
):

    product = asin_lookup.get(
        str(asin)
    )


    if product is None:

        return {
            "product_type_match": False,
            "brand_match": False,
            "budget_match": None,
            "feature_match_ratio": 0.0,
            "feature_status": "no_match",
            "support_level": "unsupported"
        }


    title = str(
        product.get(
            "title",
            ""
        )
    )


    product_type_match = (
        check_product_type_match(
            structured_query.product_type,
            title
        )
    )


    brand_match = (
        check_brand_match(
            structured_query.brand,
            title
        )
    )


    price = normalise_price(
        product.get(
            "price"
        )
    )


    budget_match = None


    if (
        structured_query.budget
        is not None
        and price is not None
    ):

        budget_match = (
            price
            <= float(
                structured_query.budget
            )
        )


    (
        feature_status,
        feature_ratio,
        matched,
        missing
    ) = check_feature_match(
        structured_query.features,
        product
    )


    hard_constraints_ok = (
        product_type_match
        and brand_match
        and budget_match is not False
    )


    if not hard_constraints_ok:

        support_level = (
            "unsupported"
        )

    elif feature_status in {
        "full_match",
        "not_requested"
    }:

        support_level = (
            "grounded"
        )

    elif feature_status == (
        "partial_match"
    ):

        support_level = (
            "partial"
        )

    else:

        support_level = (
            "unsupported"
        )


    return {

        "product_type_match":
            product_type_match,

        "brand_match":
            brand_match,

        "budget_match":
            budget_match,

        "feature_match_ratio":
            feature_ratio,

        "feature_status":
            feature_status,

        "support_level":
            support_level
    }

In [7]:
SYSTEMS = {

    "Single Agent":
        single,

    "Multi-Agent Without Coordination":
        no_coord,

    "Coordinated Multi-Agent":
        coord
}


quality_rows = []


for system_name, dataframe in SYSTEMS.items():

    for _, row in dataframe.iterrows():

        query = query_agent.parse(
            row[
                "user_query"
            ]
        )


        recommended_asin = row.get(
            "recommended_asin"
        )


        has_recommendation = (
            pd.notna(
                recommended_asin
            )
            and str(
                recommended_asin
            ).strip()
            not in {
                "",
                "None",
                "nan"
            }
        )


        if has_recommendation:

            metrics = evaluate_product(
                query,
                str(
                    recommended_asin
                )
            )

        else:

            metrics = {
                "product_type_match":
                    None,

                "brand_match":
                    None,

                "budget_match":
                    None,

                "feature_match_ratio":
                    None,

                "feature_status":
                    "no_recommendation",

                "support_level":
                    "no_recommendation"
            }


        benchmark_row = (
            benchmark[
                benchmark[
                    "test_id"
                ]
                == row[
                    "test_id"
                ]
            ]
            .iloc[0]
        )


        expected_outcome = (
            benchmark_row[
                "expected_outcome"
            ]
        )


        correct_abstention = (
            expected_outcome
            == "no_match"
            and not has_recommendation
        )


        incorrect_abstention = (
            expected_outcome
            == "match"
            and not has_recommendation
        )


        quality_rows.append({

            "system":
                system_name,

            "test_id":
                row["test_id"],

            "category":
                row["category"],

            "user_query":
                row["user_query"],

            "expected_outcome":
                expected_outcome,

            "has_recommendation":
                has_recommendation,

            "recommended_asin":
                recommended_asin,

            "product_type_match":
                metrics[
                    "product_type_match"
                ],

            "brand_match":
                metrics[
                    "brand_match"
                ],

            "budget_match":
                metrics[
                    "budget_match"
                ],

            "feature_match_ratio":
                metrics[
                    "feature_match_ratio"
                ],

            "feature_status":
                metrics[
                    "feature_status"
                ],

            "support_level":
                metrics[
                    "support_level"
                ],

            "correct_abstention":
                correct_abstention,

            "incorrect_abstention":
                incorrect_abstention,

            "total_latency":
                row[
                    "total_latency"
                ],

            "total_tokens":
                row[
                    "total_tokens"
                ],

            "estimated_cost_usd":
                row[
                    "estimated_cost_usd"
                ]
        })


quality_df = pd.DataFrame(
    quality_rows
)


display(
    quality_df.head(
        20
    )
)

,system,test_id,category,user_query,expected_outcome,has_recommendation,recommended_asin,product_type_match,brand_match,budget_match,feature_match_ratio,feature_status,support_level,correct_abstention,incorrect_abstention,total_latency,total_tokens,estimated_cost_usd
0,Single Agent,1,Brand Only,Recommend a Samsung phone.,match,True,B09C6N8P6Y,True,True,None,1.0,not_requested,grounded,False,False,8.4858,3536,0.000714
1,Single Agent,2,Brand Only,Recommend an Apple phone.,match,True,B00CX0OZHY,True,True,None,1.0,not_requested,grounded,False,False,5.9017,3135,0.000612
2,Single Agent,3,Brand Only,Suggest a Motorola smartphone.,match,True,B002VRO83K,True,True,None,1.0,not_requested,grounded,False,False,4.7732,2773,0.000561
3,Single Agent,4,Brand Only,Recommend a Nokia phone.,match,True,B003X26SLM,True,True,None,1.0,not_requested,grounded,False,False,6.3362,3137,0.000648
4,Single Agent,5,Brand Only,Suggest a Google phone.,match,True,B0B6PV17MC,False,True,None,1.0,not_requested,unsupported,False,False,5.3562,3128,0.000620
5,Single Agent,6,Brand + Feature,Recommend a Samsung phone with a good camera.,match,True,B09S6VKCLX,True,True,None,1.0,full_match,grounded,False,False,4.9867,3427,0.000666
6,Single Agent,7,Brand + Feature,Recommend a Samsung phone with long battery life.,match,True,B09WT8N5X7,True,True,None,1.0,full_match,grounded,False,False,4.1569,3318,0.000597
7,Single Agent,8,Brand + Feature,Recommend an Apple phone with a good camera.,match,True,B078WZX9LD,True,True,None,1.0,full_match,grounded,False,False,2.6330,3167,0.000546
8,Single Agent,9,Brand + Feature,Suggest a Motorola phone with a good camera.,match,True,B004P551BE,True,True,None,1.0,full_match,grounded,False,False,5.9571,2906,0.000594
9,Single Agent,10,Brand + Feature,Recommend a Samsung phone with fast performance.,match,True,B09WT8N5X7,True,True,None,0.0,no_match,unsupported,False,False,5.8461,3416,0.000706


In [8]:
summary_rows = []


for system_name in SYSTEMS.keys():

    df = quality_df[
        quality_df[
            "system"
        ]
        == system_name
    ]


    recommendations = df[
        df[
            "has_recommendation"
        ]
        == True
    ]


    no_match_cases = df[
        df[
            "expected_outcome"
        ]
        == "no_match"
    ]


    grounded = (
        recommendations[
            "support_level"
        ]
        .eq(
            "grounded"
        )
        .mean()
        if len(recommendations)
        else 0
    )


    partial = (
        recommendations[
            "support_level"
        ]
        .eq(
            "partial"
        )
        .mean()
        if len(recommendations)
        else 0
    )


    hallucination = (
        recommendations[
            "support_level"
        ]
        .eq(
            "unsupported"
        )
        .mean()
        if len(recommendations)
        else 0
    )


    abstention_accuracy = (
        no_match_cases[
            "correct_abstention"
        ].mean()
        if len(no_match_cases)
        else np.nan
    )


    feature_values = (
        recommendations[
            "feature_match_ratio"
        ]
        .dropna()
    )


    summary_rows.append({

        "system":
            system_name,

        "queries":
            len(df),

        "recommendation_rate":
            df[
                "has_recommendation"
            ].mean(),

        "mean_feature_satisfaction":
            feature_values.mean()
            if len(feature_values)
            else np.nan,

        "grounded_recommendation_rate":
            grounded,

        "partial_grounding_rate":
            partial,

        "hallucination_rate":
            hallucination,

        "correct_abstention_rate":
            abstention_accuracy,

        "mean_latency":
            df[
                "total_latency"
            ].mean(),

        "median_latency":
            df[
                "total_latency"
            ].median(),

        "mean_tokens":
            df[
                "total_tokens"
            ].mean(),

        "total_tokens":
            df[
                "total_tokens"
            ].sum(),

        "mean_cost_usd":
            df[
                "estimated_cost_usd"
            ].mean(),

        "total_cost_usd":
            df[
                "estimated_cost_usd"
            ].sum()
    })


final_summary = pd.DataFrame(
    summary_rows
)


display(
    final_summary
)

,system,queries,recommendation_rate,mean_feature_satisfaction,grounded_recommendation_rate,partial_grounding_rate,hallucination_rate,correct_abstention_rate,mean_latency,median_latency,mean_tokens,total_tokens,mean_cost_usd,total_cost_usd
0,Single Agent,60,0.816667,0.778912,0.612245,0.102041,0.285714,0.666667,6.270542,5.53905,3374.583333,202475,0.000646,0.038733
1,Multi-Agent Without Coordination,60,0.933333,0.744048,0.678571,0.125000,0.196429,0.250000,6.676695,4.40220,1847.816667,110869,0.000365,0.021904
2,Coordinated Multi-Agent,60,0.716667,0.910853,0.837209,0.139535,0.023256,0.916667,4.789863,3.79430,1851.383333,111083,0.000368,0.022080


In [9]:
quality_df.to_csv(
    FINAL_FOLDER
    / "final_query_level_evaluation.csv",
    index=False
)


final_summary.to_csv(
    FINAL_FOLDER
    / "FINAL_SYSTEM_SUMMARY.csv",
    index=False
)


print(
    "Final system evaluation saved."
)

Final system evaluation saved.


#### Precision@5 and Recall@5

In [10]:
def relevant_asins_for_query(
    user_query
):

    query = query_agent.parse(
        user_query
    )


    relevant = []


    for asin in asin_lookup.keys():

        result = evaluate_product(
            query,
            asin
        )


        if result[
            "support_level"
        ] == "grounded":

            relevant.append(
                asin
            )


    return set(
        relevant
    )

In [11]:
retrieval_rows = []


for system_name, dataframe in SYSTEMS.items():

    for _, row in dataframe.iterrows():

        retrieved_top5 = set(
            split_asins(
                row[
                    "top_5_asins"
                ]
            )[:5]
        )


        relevant = (
            relevant_asins_for_query(
                row[
                    "user_query"
                ]
            )
        )


        intersection = (
            retrieved_top5
            .intersection(
                relevant
            )
        )


        precision_at_5 = (
            len(intersection)
            / 5
        )


        recall_at_5 = (
            len(intersection)
            / len(relevant)
            if len(relevant)
            else np.nan
        )


        retrieval_rows.append({

            "system":
                system_name,

            "test_id":
                row[
                    "test_id"
                ],

            "precision_at_5":
                precision_at_5,

            "recall_at_5":
                recall_at_5,

            "relevant_products":
                len(
                    relevant
                )
        })


retrieval_metrics = pd.DataFrame(
    retrieval_rows
)


display(
    retrieval_metrics
)

,system,test_id,precision_at_5,recall_at_5,relevant_products
0,Single Agent,1,1.0,0.054945,91
1,Single Agent,2,0.8,0.250000,16
2,Single Agent,3,0.8,0.235294,17
3,Single Agent,4,0.2,0.166667,6
4,Single Agent,5,0.8,0.444444,9
...,...,...,...,...,...
175,Coordinated Multi-Agent,56,0.8,0.266667,15
176,Coordinated Multi-Agent,57,0.0,NaN,0
177,Coordinated Multi-Agent,58,0.0,NaN,0
178,Coordinated Multi-Agent,59,0.0,NaN,0


In [12]:
retrieval_summary = (
    retrieval_metrics
    .groupby(
        "system",
        as_index=False
    )
    .agg(
        mean_precision_at_5=(
            "precision_at_5",
            "mean"
        ),

        mean_recall_at_5=(
            "recall_at_5",
            "mean"
        )
    )
)


display(
    retrieval_summary
)


retrieval_metrics.to_csv(
    FINAL_FOLDER
    / "final_precision_recall_query_level.csv",
    index=False
)


retrieval_summary.to_csv(
    FINAL_FOLDER
    / "final_precision_recall_summary.csv",
    index=False
)

,system,mean_precision_at_5,mean_recall_at_5
0,Coordinated Multi-Agent,0.48,0.196668
1,Multi-Agent Without Coordination,0.48,0.196668
2,Single Agent,0.32,0.119882


#### SCALABLITIY EXPERIMENT

In [13]:
SCALABILITY_QUERY_IDS = [
    6,
    14,
    21,
    29,
    34,
    37,
    48,
    54
]


scalability_queries = (
    benchmark[
        benchmark[
            "test_id"
        ].isin(
            SCALABILITY_QUERY_IDS
        )
    ]
    .copy()
)


display(
    scalability_queries
)

,test_id,category,user_query,expected_brand,budget,expected_features,expected_outcome
5,6,Brand + Feature,Recommend a Samsung phone with a good camera.,Samsung,NaN,camera,match
13,14,Budget,Recommend a smartphone under 200.,NaN,200.0,NaN,match
20,21,Feature Only,Recommend a phone with a great camera.,NaN,NaN,camera,match
28,29,Multiple Features,Recommend a phone with a good camera and long ...,NaN,NaN,camera|long battery life,match
33,34,Multiple Features,"Recommend a smartphone with a good camera, lon...",NaN,NaN,camera|long battery life|fast performance,match
36,37,Complex,Recommend a Samsung phone under 300 with a goo...,Samsung,300.0,camera,match
47,48,Unsupported,Recommend an Apple phone with stylus support.,Apple,NaN,stylus support,no_match
53,54,No Match,Recommend a Nokia phone with an 8K camera and ...,Nokia,NaN,8k camera|1tb storage,no_match


In [15]:
SCALABILITY_QUERY_IDS = [
    6,
    14,
    21,
    29,
    34,
    37,
    48,
    54
]


scalability_queries = (
    benchmark[
        benchmark[
            "test_id"
        ].isin(
            SCALABILITY_QUERY_IDS
        )
    ]
    .copy()
)


display(
    scalability_queries
)

,test_id,category,user_query,expected_brand,budget,expected_features,expected_outcome
5,6,Brand + Feature,Recommend a Samsung phone with a good camera.,Samsung,NaN,camera,match
13,14,Budget,Recommend a smartphone under 200.,NaN,200.0,NaN,match
20,21,Feature Only,Recommend a phone with a great camera.,NaN,NaN,camera,match
28,29,Multiple Features,Recommend a phone with a good camera and long ...,NaN,NaN,camera|long battery life,match
33,34,Multiple Features,"Recommend a smartphone with a good camera, lon...",NaN,NaN,camera|long battery life|fast performance,match
36,37,Complex,Recommend a Samsung phone under 300 with a goo...,Samsung,300.0,camera,match
47,48,Unsupported,Recommend an Apple phone with stylus support.,Apple,NaN,stylus support,no_match
53,54,No Match,Recommend a Nokia phone with an 8K camera and ...,Nokia,NaN,8k camera|1tb storage,no_match


In [14]:
K_VALUES = [
    5,
    10,
    20,
    30
]

In [17]:
# =====================================================
# COMBINE SCALABILITY RESULTS
# =====================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()

FINAL_FOLDER = (
    PROJECT_ROOT
    / "Results"
    / "Final"
)

no_coord_scalability = pd.read_csv(
    FINAL_FOLDER
    / "no_coord_scalability.csv"
)

coordinated_scalability = pd.read_csv(
    FINAL_FOLDER
    / "coordinated_scalability.csv"
)

scalability_all = pd.concat(
    [
        no_coord_scalability,
        coordinated_scalability
    ],
    ignore_index=True
)

scalability_summary = (
    scalability_all
    .groupby(
        [
            "system",
            "retrieval_k"
        ],
        as_index=False
    )
    .agg(
        mean_latency=(
            "latency",
            "mean"
        ),
        std_latency=(
            "latency",
            "std"
        ),
        mean_tokens=(
            "tokens",
            "mean"
        ),
        std_tokens=(
            "tokens",
            "std"
        )
    )
)

display(
    scalability_summary
)

scalability_summary.to_csv(
    FINAL_FOLDER
    / "final_scalability_summary.csv",
    index=False
)

print(
    "Scalability summary saved."
)

,system,retrieval_k,mean_latency,std_latency,mean_tokens,std_tokens
0,Coordinated Multi-Agent,5,4.131550,0.603951,1650.625,129.525439
1,Coordinated Multi-Agent,10,4.080650,0.395685,1638.500,131.657564
2,Coordinated Multi-Agent,20,6.258025,1.796878,1969.500,393.117939
3,Coordinated Multi-Agent,30,6.496462,2.823595,2015.375,451.632258
4,Multi-Agent Without Coordination,5,3.987600,0.647407,1654.250,124.983142
5,Multi-Agent Without Coordination,10,3.497663,0.268361,1644.125,134.728762
6,Multi-Agent Without Coordination,20,5.630175,3.101755,2017.250,481.862088
7,Multi-Agent Without Coordination,30,5.169837,1.992185,2000.750,431.849097


Scalability summary saved.
